In [6]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

BPM = 100
N_BARS = 8

# Per sound one row where you can write a x for a hit and a . as a rest, ervyry step is a 16th note
# 16 characters = one 4/4 bar
PATTERNS = {
    "drip":    "................",
    "knock":   "....x.......x...",
    "pebble":  "..x...x...x..x..",
    "rustle":  "x...x...x...x...",
    "cricket": "..x.......x.....",
}


SR = 22050
rng = np.random.default_rng(1)

def time(dur):
    return np.arange(int(dur * SR)) / SR

def noise(n):
    return np.diff(rng.normal(size=n + 1))

# the five percussion sounds
def drip():
    t = time(0.25)
    f0 = rng.uniform(200, 350)
    phase = 2 * np.pi * f0 * (t + 8 * t**2 / 2)
    return np.sin(phase) * np.exp(-22 * t) * np.minimum(t / 0.002, 1)

def knock():
    t = time(0.15)
    tone = np.sin(2 * np.pi * rng.uniform(600, 620) * t) * np.exp(-t / 0.025)
    snap = noise(len(t)) * np.exp(-t / 0.004)
    return tone + 0.7 * snap

def pebble():
    t = time(0.06)
    tone = np.sin(2 * np.pi * rng.uniform(2000, 3500) * t)
    return (0.5 * tone + 0.5 * noise(len(t))) * np.exp(-t / 0.006)

def rustle():
    t = time(0.12)
    return noise(len(t)) * np.hanning(len(t))

def cricket():
    y = np.zeros(int(0.12 * SR))
    t = time(0.012)
    f = rng.uniform(4000, 5000)
    for p in range(3):
        a = int(p * 0.025 * SR)
        y[a:a + len(t)] += np.sin(2 * np.pi * f * t) * np.hanning(len(t))
    return y

# sound function and volume for each row
KIT = {"drip": (drip, 1.0), "knock": (knock, 0.8), "pebble": (pebble, 0.4),
       "rustle": (rustle, 0.3), "cricket": (cricket, 0.35)}

# building the beat
steps = len(PATTERNS["drip"])                  # length of the pattern as steps per bar
assert all(len(p) == steps for p in PATTERNS.values()), "all patterns must have the same length"
step_dur = 60 / BPM / 4                        # seconds per 16th note
mix = np.zeros(int((N_BARS * steps * step_dur + 1) * SR))   # silent track

for bar in range(N_BARS):
    for name, pattern in PATTERNS.items():
        make, vol = KIT[name]
        for s, char in enumerate(pattern):
            if char != "x":
                continue
            sound = make()
            sound = sound / np.abs(sound).max()                   # same loudness for every sound
            t = (bar * steps + s) * step_dur + rng.normal(0, 0.008)   # time + tiny human wobble
            start = max(int(t * SR), 0)
            mix[start:start + len(sound)] += sound * vol * rng.uniform(0.7, 1.0)

# a little bit of echo
d = int(0.12 * SR)
mix = mix + 0.3 * np.concatenate([np.zeros(d), mix[:-d]])

mix = mix / np.abs(mix).max()
sf.write("nature_beat.wav", mix, SR)
display(Audio(mix, rate=SR))